# 04 - Compare CU results from the original and preprocessed scan
The input is always the scanned PDF configured in `.env`. Compare it unchanged against full-page rendering and optional filters, with the same Content Understanding analyzer in every case. No intake-image comparison or notebook 01 is required.

Returned business fields are shown when the selected analyzer supplies them. The labelled amount/sign checks on CU Markdown are reported separately and must not be mistaken for business-field validation. Only the local expected-values CSV supplies ground truth.

All main variants preserve the complete scan's page coverage. A redaction candidate is a separate, optional single-page test and is not automatically safe to share.

All results and screenshots are local under ignored `output/`. The tracked notebook remains free of private outputs.

In [ ]:
import hashlib, html, json, re
from decimal import Decimal
from pathlib import Path
import cv2 as cv
import numpy as np
import pandas as pd
import pypdfium2 as pdfium
from PIL import Image, ImageDraw, ImageFilter
from dotenv import dotenv_values

OCR = Path('..').resolve()
if not (OCR / '.env').is_file():
    raise FileNotFoundError('Copy demo/ocr/.env.example to demo/ocr/.env first.')
config = dotenv_values(OCR / '.env')
PDF = (OCR / config['OCR_INPUT_FILE']).resolve()
PAGE = int(config.get('OCR_FOCUS_PAGE', 1))
DPI = int(config.get('OCR_TARGET_DPI', 300))
ANALYZER = (config.get('CU_ANALYZER_ID') or '').strip()
API = config.get('CU_API_VERSION') or '2025-11-01'
OUT = OCR / 'output'
(OUT / 'versions').mkdir(parents=True, exist_ok=True)
(OUT / 'screenshots').mkdir(exist_ok=True)
original_hash = hashlib.sha256(PDF.read_bytes()).hexdigest()
expected = pd.read_csv(OCR / config['OCR_EXPECTED_AMOUNTS'], dtype={'amount': str})
display(expected[['page', 'label', 'amount']])

## Build controlled inputs independently
Reuses the PDFium scale and lossless multi-page TIFF pattern from the [existing renderer](https://github.com/jomedinagomez/azure-ai-foundry-insurance/blob/f6949572c5aaa6bc77b1ac592a92affc62d98346/demo/sov/preprocess/pdf_render.py), [Pillow filters](https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html), and [OpenCV non-local means](https://docs.opencv.org/4.x/d5/d69/tutorial_py_non_local_means.html). Parameters match notebooks 02/03.

Every derivative starts from the complete scanned PDF page, including its visible layers. TIFF is only the preprocessed transport format accepted by CU; it is not a different incoming document or a different extraction engine.

In [ ]:
def save_lossless(pages, path, dpi):
    if path.suffix.casefold() not in ('.tif', '.tiff', '.png'):
        raise ValueError('Use TIFF or PNG; JPEG output is intentionally refused.')
    if path.suffix.casefold() == '.png' and len(pages) != 1:
        raise ValueError('Use TIFF for multiple pages.')
    if path.exists():
        with Image.open(path) as saved:
            identical = saved.n_frames == len(pages)
            if identical:
                for index, page in enumerate(pages):
                    saved.seek(index)
                    if saved.mode != page.mode or saved.size != page.size or saved.tobytes() != page.tobytes() or tuple(saved.info.get('dpi', ())) != tuple(dpi):
                        identical = False
                        break
            if identical:
                return
    compression = 'group4' if all(page.mode == '1' for page in pages) else 'tiff_lzw'
    options = {'compression': compression} if path.suffix.casefold() != '.png' else {}
    pages[0].save(path, save_all=len(pages) > 1, append_images=pages[1:], dpi=dpi, **options)

base = []
with pdfium.PdfDocument(str(PDF)) as document:
    page_count = len(document)
    for page in document:
        bitmap = page.render(scale=DPI / 72)
        try:
            base.append(bitmap.to_pil().convert('L'))
        finally:
            bitmap.close()
median = [page.filter(ImageFilter.MedianFilter(3)) for page in base]
variants = {
    f'rendered_{DPI}dpi': base,
    f'median_{DPI}dpi': median,
    f'nlmeans_{DPI}dpi': [Image.fromarray(cv.fastNlMeansDenoising(np.asarray(page), None, h=3, templateWindowSize=7, searchWindowSize=21)) for page in base],
    f'sharpened_{DPI}dpi': [page.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=2)) for page in base],
    f'denoise_sharpen_{DPI}dpi': [page.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=2)) for page in median],
    f'thickened_{DPI}dpi': [Image.fromarray(cv.erode(np.asarray(page), np.ones((2, 1), np.uint8))) for page in base],
}
inputs = {'original_pdf': PDF}
for label, pages in variants.items():
    path = OUT / 'versions' / f'{label}.tiff'
    save_lossless(pages, path, (DPI, DPI))
    inputs[label] = path
print('Full-document variants:', len(inputs), '| pages per variant:', page_count)

## Row-specific amount and sign checks
Money tokens accept leading ASCII/Unicode minus, parentheses, trailing minus and a CR suffix. An expected value must match the amount on the configured page and labelled row, not somewhere else in the document. CU's PageBreak markers delimit the pages; an absent page is not searched globally. `kept` means exact digits and sign; `lost` means the correct magnitude was found without its expected negative sign. `wrong sign`, `not found` and `ambiguous` are not passing results. The parser is an evaluation aid, not a financial-value correction algorithm.

HTML table rows are kept together. If a complete label spans two adjacent rows, ordered label words may match across that pair; this is reported as an adjacent-row association, not a missing digit. Plain Markdown uses a short local window. Review saved context when labels or digits are not found. A missing label is never invented from ground truth.

In [ ]:
money_pattern = re.compile(r'(?<![\d.])(?:\([ \t]*)?(?:[-\u2212\u2013][ \t]*)?\$?[ \t]*(?:[-\u2212\u2013][ \t]*)?(?:\d{1,3}(?:,\d{3})+|\d+)\.\d{2}(?:[ \t]*[-\u2212\u2013]|[ \t]*CR|[ \t]*\))?(?!\d)', re.I)

def find_signed_amounts(text):
    found = []
    for match in money_pattern.finditer(text):
        token = match.group().strip()
        negative = any(mark in token for mark in ('-', '\u2212', '\u2013', '(')) or token.upper().endswith('CR')
        value = Decimal(re.sub(r'[^\d.]', '', token))
        found.append({'text': token, 'value': -value if negative else value})
    return found

def normalize(text):
    return ' '.join(re.sub(r'[^a-z0-9]+', ' ', html.unescape(re.sub(r'<[^>]+>', ' ', text)).casefold()).split())

def compare_signs(expected_rows, text):
    page_texts = re.split(r'<!--\s*PageBreak\s*-->', text, flags=re.I)
    rows = []
    for item in expected_rows.to_dict('records'):
        page_index = int(item['page']) - 1
        page_text = page_texts[page_index] if 0 <= page_index < len(page_texts) else ''
        html_rows = re.findall(r'<tr\b[^>]*>.*?</tr>', page_text, re.S | re.I)
        lines = page_text.splitlines()
        blocks = html_rows if html_rows else [' '.join(lines[i:i + 3]) for i in range(len(lines))]
        want = Decimal(str(item['amount']))
        contexts = [block for block in blocks if normalize(item['label']) in normalize(block)]
        association = 'single row'
        if not contexts and html_rows:
            label_pattern = r'\b' + r'\b.*?\b'.join(re.escape(word) for word in normalize(item['label']).split()) + r'\b'
            pairs = [' '.join(html_rows[i:i + 2]) for i in range(len(html_rows) - 1)]
            contexts = [block for block in pairs if re.search(label_pattern, normalize(block))]
            association = 'adjacent rows' if contexts else 'label not found'
        amounts = [token for block in contexts for token in find_signed_amounts(html.unescape(re.sub(r'<[^>]+>', ' ', block)))]
        same_magnitude = {token['value'] for token in amounts if abs(token['value']) == abs(want)}
        if len(same_magnitude) > 1:
            status = 'ambiguous'
        elif want in same_magnitude:
            status = 'kept'
        elif same_magnitude:
            status = 'lost' if want < 0 else 'wrong sign'
        else:
            status = 'not found'
        observed = ', '.join(sorted({token['text'] for token in amounts if abs(token['value']) == abs(want)}))
        rows.append({'page': item['page'], 'label': item['label'], 'expected': str(want), 'observed': observed, 'status': status, 'association': association})
    return pd.DataFrame(rows)

## Reuse the official SDK request and the experiment cache
The client, `begin_analyze_binary` and polling follow the [official binary-analysis sample](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py). [MIT notice](../LICENSE). The same endpoint, analyzer, API and input hash identify a cached response. No expected values are sent to the service. Missing endpoint configuration is explicitly skipped; other service errors are raised.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

endpoint = config.get('CONTENTUNDERSTANDING_ENDPOINT', '').rstrip('/')
if endpoint and not ANALYZER:
    raise ValueError('Set CU_ANALYZER_ID in demo/ocr/.env to the existing analyzer used for extraction.')
key = config.get('CONTENTUNDERSTANDING_KEY')
client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key) if key else DefaultAzureCredential(process_timeout=60), api_version=API) if endpoint else None

def analyze(path, label):
    if client is None:
        print('Skipped: set CONTENTUNDERSTANDING_ENDPOINT in demo/ocr/.env')
        return None
    data = path.read_bytes()
    digest = hashlib.sha256(data + f'{endpoint}|{ANALYZER}|{API}'.encode()).hexdigest()
    cache = OUT / 'cu-cache' / f'{digest}.json'
    cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        result = json.loads(cache.read_text(encoding='utf-8'))
        print(label, '- cached')
    else:
        print(label, '- submitting to', ANALYZER)
        result = client.begin_analyze_binary(analyzer_id=ANALYZER, binary_input=data).result().as_dict()
        cache.write_text(json.dumps(result, indent=2), encoding='utf-8')
    text = '\n\n'.join(item.get('markdown', '') for item in result['contents'])
    (OUT / f'{label}.md').write_text(text, encoding='utf-8')
    (OUT / f'{label}.input.json').write_text(json.dumps({'input_sha256': hashlib.sha256(data).hexdigest(), 'cache': cache.name, 'analyzer': ANALYZER, 'api_version': API}, indent=2), encoding='utf-8')
    print('Input file:', path.resolve())
    print('Full CU response JSON:', cache.resolve())
    print('Full CU Markdown:', (OUT / f'{label}.md').resolve())
    return result

responses, details, summary = {}, [], []
for label, path in inputs.items():
    result = analyze(path, label)
    responses[label] = result
    if result is None:
        summary.append({'case': label, 'availability': 'not analyzed'})
        continue
    text = '\n\n'.join(item.get('markdown', '') for item in result['contents'])
    checks = compare_signs(expected, text)
    checks.insert(0, 'case', label)
    details.append(checks)
    counts = checks['status'].value_counts()
    field_count = sum(len(item.get('fields', {})) for item in result['contents'])
    summary.append({'case': label, 'availability': 'analyzed', 'analyzer': ANALYZER, 'structured_fields': field_count, 'measurement': 'CU Markdown amount/sign checks', 'input_pages': page_count, 'expected': len(checks),
                    **{name: int(counts.get(name, 0)) for name in ('kept', 'lost', 'wrong sign', 'not found', 'ambiguous')}})
comparison = pd.DataFrame(summary)
detail = pd.concat(details, ignore_index=True) if details else pd.DataFrame()
comparison.to_csv(OUT / 'results.csv', index=False)
detail.to_csv(OUT / 'amount_details.csv', index=False)
display(comparison)
if not detail.empty:
    display(detail[detail['label'].str.contains(config.get('OCR_KEYWORD', 'property tax'), case=False, regex=False)])

## Inspect CU-extracted fields separately
The configured analyzer is unchanged across inputs. The following table previews its returned typed fields; complete values remain in the saved responses. No business schema, field mapping or corrected sign is invented. If the analyzer returns no fields, this is explicitly an OCR/Markdown comparison only. Set `CU_ANALYZER_ID` to the actual existing extraction analyzer before interpreting business-field results.

In [ ]:
fields_by_case = {label: [item.get('fields', {}) for item in result['contents']] for label, result in responses.items() if result is not None}
(OUT / 'cu_fields_by_case.json').write_text(json.dumps(fields_by_case, indent=2), encoding='utf-8')
field_previews = {}
for label, contents in fields_by_case.items():
    values = {}
    for index, fields in enumerate(contents, 1):
        for name, field in fields.items():
            value = next((value for key, value in field.items() if key.startswith('value')), None)
            values[f'{index}: {name}'] = json.dumps(value, ensure_ascii=False)[:180]
    if values:
        field_previews[label] = values
if field_previews:
    display(pd.DataFrame(field_previews))
else:
    print('No structured fields returned. The current analyzer provides OCR/layout only; these scores do not validate payout-field extraction.')

## Optional local redaction candidate - not automatically shareable
Read manually selected black-out boxes from the ignored local JSON. The boxes use inches and apply to one rendered page. This is pixel-level editing with [Pillow ImageDraw.rectangle](https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html#PIL.ImageDraw.ImageDraw.rectangle), not a PDF re-save. The result is a single-page image, so only that page's expected values are checked.

A configured candidate is analyzed with the same CU endpoint. Do not send it elsewhere without human privacy review. It is an original-bug reproduction only if both the original and redacted inputs still lose the relevant sign. If the original passes or redaction changes the outcome, say so explicitly.

In [ ]:
redaction_file = OCR / config.get('OCR_REDACT_BOXES', 'input/redact_boxes.json')
redaction_checks = pd.DataFrame()
if redaction_file.is_file():
    spec = json.loads(redaction_file.read_text(encoding='utf-8'))
    candidate = base[int(spec['page']) - 1].copy()
    draw = ImageDraw.Draw(candidate)
    for box in spec['boxes']:
        draw.rectangle(tuple(round(v * DPI) for v in box), fill=0)
    path = OUT / 'versions' / 'redacted_candidate.tiff'
    save_lossless([candidate], path, (DPI, DPI))
    candidate.save(OUT / 'screenshots' / 'redacted_candidate.png', dpi=(DPI, DPI))
    result = analyze(path, 'redacted_candidate')
    if result is not None:
        text = '\n\n'.join(item.get('markdown', '') for item in result['contents'])
        candidate_expected = expected[expected['page'] == int(spec['page'])].copy()
        candidate_expected['page'] = 1
        redaction_checks = compare_signs(candidate_expected, text)
        redaction_checks['source_page'] = int(spec['page'])
        redaction_checks.to_csv(OUT / 'redacted_amount_details.csv', index=False)
        counts = redaction_checks['status'].value_counts()
        candidate_summary = {'case': 'redacted_candidate', 'availability': 'analyzed - single page', 'analyzer': ANALYZER, 'structured_fields': sum(len(item.get('fields', {})) for item in result['contents']), 'measurement': 'CU Markdown amount/sign checks', 'input_pages': 1, 'expected': len(redaction_checks), **{name: int(counts.get(name, 0)) for name in ('kept', 'lost', 'wrong sign', 'not found', 'ambiguous')}}
        comparison = pd.concat([comparison, pd.DataFrame([candidate_summary])], ignore_index=True)
        comparison.to_csv(OUT / 'results.csv', index=False)
        display(redaction_checks)
        original_lost = set(detail.loc[(detail['case'] == 'original_pdf') & (detail['status'] == 'lost'), 'label'])
        candidate_lost = set(redaction_checks.loc[redaction_checks['status'] == 'lost', 'label'])
        print('Same lost-sign labels in original and candidate:', sorted(original_lost & candidate_lost))
        print('Human privacy review is still required. No file was published or shared outside the configured CU resource.')
    candidate.close()
else:
    print('No redaction boxes configured: candidate not created or analyzed.')

## Local visual report
Save the comparison next to the full-page previews. Read `not found` and `ambiguous` rows before drawing conclusions. Visual appearance is not an OCR score. The supplied expected values cover selected financial rows, not every word on the page.

The report and executed copy stay under ignored output. The original source hash is checked again. No material is committed, pushed or uploaded to a third-party OCR service.

In [ ]:
report_parts = [
    '<!doctype html><meta charset="utf-8"><title>Local OCR comparison</title>',
    '<style>body{font:16px Segoe UI,Arial;margin:28px;color:#142334}table{border-collapse:collapse;font-size:14px}th,td{padding:7px;border:1px solid #ccd5de}th{background:#e8eff6}img{max-width:100%;height:auto}.note{background:#fff5d6;padding:14px}</style>',
    '<h1>OCR input comparison</h1>',
    '<p class="note">One scanned PDF, one configured Content Understanding analyzer. Compare the original with preprocessed versions of the same pages. Markdown sign checks and returned structured fields are separate measurements.</p>',
    '<section id="summary"><h2>Row-specific results</h2>' + comparison.fillna('-').to_html(index=False, escape=True) + '</section>',
]
if field_previews:
    report_parts.append('<h2>CU extracted fields</h2>' + pd.DataFrame(field_previews).fillna('-').to_html(escape=True))
else:
    report_parts.append('<p class="note">The configured analyzer returned no structured fields. This report validates OCR/Markdown checks only, not payout-field extraction.</p>')
if not detail.empty:
    original_negative = detail[(detail['case'] == 'original_pdf') & (detail['expected'].map(Decimal) < 0)]
    if not original_negative.empty and (original_negative['status'] == 'kept').all():
        report_parts.append('<p class="note"><strong>Measured outcome:</strong> the original PDF preserved all configured negative signs in this run. The missing-sign failure was not reproduced; passing derivatives are not evidence of a fix.</p>')
    else:
        report_parts.append('<p class="note">Review the original negative-value checks before claiming a reproduction or improvement.</p>')
    focus = detail[detail['label'].str.contains(config.get('OCR_KEYWORD', 'property tax'), case=False, regex=False)]
    report_parts.append('<section id="focus"><h2>Affected amount and sign</h2>' + focus.to_html(index=False, escape=True) + '</section>')
    problems = detail[detail['status'] != 'kept']
    report_parts.append('<h2>Checks requiring review</h2>' + (problems.to_html(index=False, escape=True) if not problems.empty else '<p>All configured rows kept their amount and sign in the analyzed full-document variants.</p>'))
for filename, heading in [('render_comparison.png', 'Original and processed full-page previews'), ('filter_comparison.png', 'Full-page filter variants'), ('caution_comparison.png', 'Full-page caution-only demonstrations')]:
    if (OUT / 'screenshots' / filename).is_file():
        report_parts.append(f'<section><h2>{heading}</h2><img src="screenshots/{filename}" alt="{heading}"></section>')
report_parts.append('<p>Matched counts, pixel counts and visual appearance are not a guarantee of correct OCR on another document. Redaction candidates require human privacy review.</p>')
(OUT / 'comparison.html').write_text('\n'.join(report_parts), encoding='utf-8')
for pages in variants.values():
    for page in pages:
        page.close()
print('Report:', OUT / 'comparison.html')
print('Comparison CSV:', (OUT / 'results.csv').resolve())
print('CU fields JSON:', (OUT / 'cu_fields_by_case.json').resolve())
print('Original unchanged:', hashlib.sha256(PDF.read_bytes()).hexdigest() == original_hash)

## Takeaways
Start from the scanned PDF, keep every page, and compare the original with preprocessing through the same Content Understanding analyzer. Image libraries prepare inputs; they do not replace CU extraction. Inspect returned business fields separately from Markdown diagnostics. Use representative failing and non-failing scans before changing production. Notebook 01 is not required.